# Training A Flow (using python objects)

This is the Python-script companion to the QuickStart and Training A Flow (using CLI) tutorials. Instead of running python -m bilbyflow.scripts.train config.yaml, we walk through the same pipeline calling the package API directly — the same functions the CLI scripts orchestrate.

Why bother? Because every CLI script is just orchestration: the numerics live in the package (data.*, nn.*, training.*, inference.*), and once you can call them yourself you can swap embeddings, intercept the dataset, or run any stage in isolation. This notebook mirrors scripts/train.py step for step, so you can diff the two and see there is no magic.

The steps, in order:

1. Load the config and build the NPE (embedding + flow)
2. Build/load the banks — waveforms, sky, PSD, (optional) noise segments
3. Construct the on-the-fly datasets
4. Generate fixed val/test sets and fit the standardiser
5. Run the pre-training data diagnostics
6. Train (curriculum-driven)
7. Save, draw example posteriors, PP test
8. Evaluate reweighting efficiency (CLI, as in the QuickStart)

Prerequisites: `BilbyFlow` installed (`pip install .` from the cloned repo — see the QuickStart), and off-source noise segments downloaded if you want PSD conditioning (recommended, see [companion notebook](TrainingAFlow.html) for how to do so from cli).

## Outline

The training script runs the following steps in order:

1. Build/load banks — waveforms (intrinsic), sky (extrinsic), PSD (detector noise), and optionally a noise-segment bank
2. Construct datasets — an on-the-fly training set and a disjoint validation/test set from separate waveform banks
3. Fit the standardiser — z-scoring statistics for the input (x) and target (theta)
4. Build the NPE — embedding + flow, with the prior box set from the standardised prior bounds
5. Run diagnostics — whitening checks, PSD stats, sample inspection
6. Train — curriculum-driven, with optional auxiliary supervision and JEPA consistency
7. Save and evaluate — model weights, example posteriors, PP test

Each step is discussed below, but the important thing is that all of this is driven by one YAML config file. The script itself is just handling a bunch of internals — the numerics live in the package modules, and the config parameterises them. Most functions/classes should be fairly modular and can be swapped out as wanted. The tutorials after this detail more custom choices.

## 0. Setup and config

Everything is driven by one YAML config — see the CLI tutorial for the full annotated key reference. `load_config` handles both a path to a `.yaml` and a run directory containing `config.yaml`. 

First, we'll download the scripts folder so we have access to some example scripts. This is run from terminal.

```bash
python -m bilbyflow.scripts.init_scripts --list
```
```text
  download_events.py
  download_noise.py
  fd_psd_embedding.py
  init_scripts.py
  make_injections.py
  plot_injections.py
  plot_summary.py
  reweight_injections.py
  reweight_real.py
  run_config_P100.yaml
  sk_run_config_resnet_1_A100.yaml
  train.py
  utils.py
```

Which should show you the list of files in the `scripts` folder in the package including an example config that should be usable out of the box. This can actually be downloaded by

```bash
python -m bilbyflow.scripts.init_scripts
```
```text
['download_events.py', 'download_noise.py', 'fd_psd_embedding.py', 'init_scripts.py', 'make_injections.py', 'plot_injections.py', 'plot_summary.py', 'reweight_injections.py', 'reweight_real.py', 'run_config_P100.yaml', 'sk_run_config_resnet_1_A100.yaml', 'train.py', 'utils.py']
  download_events.py -> /Users/lpin0002/Desktop/bbflow_testing/
  download_noise.py -> /Users/lpin0002/Desktop/bbflow_testing/
  fd_psd_embedding.py -> /Users/lpin0002/Desktop/bbflow_testing/
  init_scripts.py -> /Users/lpin0002/Desktop/bbflow_testing/
  make_injections.py -> /Users/lpin0002/Desktop/bbflow_testing/
  plot_injections.py -> /Users/lpin0002/Desktop/bbflow_testing/
  plot_summary.py -> /Users/lpin0002/Desktop/bbflow_testing/
  reweight_injections.py -> /Users/lpin0002/Desktop/bbflow_testing/
  reweight_real.py -> /Users/lpin0002/Desktop/bbflow_testing/
  skip run_config_P100.yaml (exists, use --overwrite)
  skip sk_run_config_resnet_1_A100.yaml (exists, use --overwrite)
  train.py -> /Users/lpin0002/Desktop/bbflow_testing/
  utils.py -> /Users/lpin0002/Desktop/bbflow_testing/

11 copied, 2 skipped -> /Users/lpin0002/Desktop/bbflow_testing
these are copies — edit freely, they import from the installed package
```

Now we can start working on the code. As stated in the companion tutorials, this will still be pretty high level (as in abstracted), later tutorials will go into how you can do more of the nitty gritty.

## Imports

In [1]:
import pickle

import numpy as np
import torch
import yaml

from bilbyflow.io.config import load_config, get_reference_detector_data
from bilbyflow.coordinates.sky import samples_detector_to_radec
from bilbyflow.coordinates.params import dL_to_physical
from bilbyflow.data.dataset import OnTheFlyGWDataset, generate_fixed_dataset
from bilbyflow.data.standardiser import check_aux_stats, check_amp_stats
from bilbyflow.data.banks import (precompute_waveforms, precompute_sky_bank,
                                  load_or_compute)
from bilbyflow.nn.aux_head import AUX_NAMES, N_AUX, AuxHead
from bilbyflow.training.trainer import custom_train_npe
from bilbyflow.plotting.training import plot_losses

In [2]:
# script-side conveniences if utils is in your working directory then you can use that one
    # in my case, I've put the example files in a folder called `bbflow_testing` in my working directory
from bbflow_testing.utils import (run_inference, plot_corner_fig, pp_test,
                   plot_aux_diagnostics, run_diagnostics,
                   print_run_banner, build_val_waveforms,
                   build_psd_bank, build_noise_bank, fit_standardiser,
                   build_npe)

## Parsing Config 

In [3]:
CONFIG_PATH = "bbflow_testing/run_config_P100.yaml"   # the example config bilbyflow-init copies over
OUT = "OUTPUTS/notebook_run"                   # output directory for this run

import os
os.makedirs(OUT, exist_ok=True) # will overwrite data if the same output/analysis path are used

cfg = load_config(CONFIG_PATH)
PARAM_NAMES = cfg["inferred_parameters"]
print_run_banner(cfg)     # prior families, curriculum stages, aux settings

# freeze the config into the run directory — the reweighting scripts read it back
with open(f"{OUT}/config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

[priors] actual families: chirp_mass:UniformInComponentsChirpMass, mass_ratio:UniformInComponentsMassRatio, luminosity_distance:PowerLaw, theta_jn:Sine, ra:Uniform, dec:Cosine, a_1:Uniform, a_2:Uniform, tilt_1:Sine, tilt_2:Sine, phi_12:Uniform, phi_jl:Uniform, phase:Uniform, geocent_time:Uniform, psi:Uniform
[priors] dL coordinate: ln(dL)
[curriculum] 7 stages, dL caps [800, 1100, 1500, 2000, 2500, 3500, 5000] Mpc
[aux] supervision ON: lambda=0.5, anneal_frac=0.7, slice=256 ctx dims, 10 targets


Note on OUT: `scripts/train.py` builds a timestamped directory via `make_out_dir` (and handles `--resume` symlinks). 

In a notebook a fixed directory is easier — every bank below is cached in OUT via `load_or_compute`, so re-running a cell loads from disk instead of recomputing. Delete the corresponding `.pkl` to force a rebuild.

## 1. Build the NPE

`build_npe` is the model factory — training and reweighting both call it, so the architecture has exactly one definition and a checkpoint always matches. It returns the model and the analysis prior box, with the sky corner already reparameterised into detector-frame coordinates (dt_HL, phi_det).

To use a custom embedding, pass `embedding_cls=YourStrainEmbeddingSubclass` here — that is the only line that differs between `scripts/train.py` and `scripts/fd_psd_embedding.py`.

In [4]:
npe, (PRIOR_LOW, PRIOR_HIGH) = build_npe(cfg)
embedding = npe.embedding
print(f"embedding: {type(embedding).__name__}, reads {embedding._blocks}, "
      f"context {embedding.context_dim}")

embedding: Conv1dResNetEmbedding, reads ['fd', 'td', 'psd'], context 512


The flow's prior box is still empty at this point — the normalised bounds need the standardiser, which doesn't exist yet. We fill it in step 3. The x packing is fixed by `data.canonical.build_x_full`, so nothing here has to agree with the dataset beyond the config grid.

## 2. Banks

Four banks, each cached in OUT:

- waveform bank — (h+, hx) pairs at intrinsic draws (seed 42; the val/test bank uses seed 4242 so intrinsics are disjoint)
- sky bank — extrinsic draws projected into (fp, fc, dt_HL, phi_det)
- PSD bank — Welch estimates over the real noise segments, era-balanced, ASD-floored (None -> fixed bilby PSD if no noise_data_dir)
- noise bank — real segments; only built when `noise_source: real` or `embed_consistency` with real rows

In [5]:
h_plus, h_cross, intr_params = load_or_compute(
    f"{OUT}/waveforms.pkl", precompute_waveforms, cfg, seed_val=42)

h_plus_v, h_cross_v, intr_params_v = load_or_compute(
    f"{OUT}/val_waveforms.pkl", build_val_waveforms, cfg)

sky_bank = load_or_compute(f"{OUT}/sky_bank.pkl", precompute_sky_bank,
                           cfg, n_sky=cfg.get("n_sky_bank", 100000))

noise_data_dir = cfg.get("noise_data_dir", None)
psd_bank = build_psd_bank(cfg, OUT, noise_data_dir)     # None -> fixed bilby PSD
noise_bank = build_noise_bank(cfg, OUT, noise_data_dir) # None unless needed

Loading cached waveforms.pkl...
Loading cached val_waveforms.pkl...
Loading cached sky_bank.pkl...
No noise_data_dir in config -- using fixed bilby default PSD


The training dataset combines the banks on the fly: each __getitem__ draws a waveform + sky position + PSD + fresh noise and packs the canonical x-vector. The val/test datasets use the disjoint waveform bank.

In [6]:
ref_data = get_reference_detector_data(cfg)

dataset = OnTheFlyGWDataset(h_plus, h_cross, intr_params, ref_data, cfg,
                            sky_bank, psd_bank=psd_bank,
                            noise_bank=noise_bank)
val_dataset = OnTheFlyGWDataset(h_plus_v, h_cross_v, intr_params_v,
                                ref_data, cfg, sky_bank,
                                psd_bank=psd_bank, noise_bank=noise_bank)

# full dL range for the fixed sets and standardiser fit
# (the curriculum applies its own caps during training)
dataset.set_dL_cap(None)
val_dataset.set_dL_cap(None)

print(f"Dataset: {len(dataset)} training waveforms | "
      f"{len(val_dataset)} disjoint val/test waveforms | "
      f"dL range [{dataset.dL_all.min():.0f}, {dataset.dL_all.max():.0f}] Mpc")

  noise_source=gaussian_physical


ValueError: psd_conditioning=True requires a PSD bank